<div style="background-color:#667eea; color:#fff; padding:22px 26px; border-radius:14px;  margin:10px 0;">
  <h2 style="margin:0 0 10px 0; color:#fff;">🧭 Regression Workflow (used in every lab)</h2>
  <ol style="margin:0; padding-left:22px; line-height:1.7;">
    <li>📦 <b>Import libraries</b></li>
    <li>📊 <b>Load the data &amp; describe the dataset / task</b></li>
    <li>🔎 <b>Exploratory Data Analysis (EDA)</b></li>
    <li>🧹 <b>Preprocess</b> — <i>split first, then standardize</i></li>
    <li>🧠 <b>Train the model</b></li>
    <li>📈 <b>Evaluate</b> (and compare with a baseline)</li>
    <li>🧩 <b>Interpret / Exercises</b></li>
  </ol>
</div>

<div style="background:#fff4e5; border-left:6px solid #ff9800; padding:14px 18px; border-radius:8px; margin:10px 0; color:#663c00;">
  <b>⚠️ Golden rule.</b> Fit any data-driven transformer (scaler, encoder, imputer…) on the <b>training set only</b>, <b>after</b> the train/test split. Fitting on the full dataset leaks information from the test fold into training.
</div>


<div style="background-color:#f093fb; color:#fff; padding:22px 26px; border-radius:14px;  margin:10px 0;">
  <h1 style="margin:0 0 6px 0; color:#fff;">📈 Lab 1 — ML Regression for Advertising Sales</h1>
  <p style="margin:0; opacity:0.95;">Linear Regression (from scratch), Ridge, and LASSO on the classic <i>Advertising</i> dataset.</p>
</div>


<div style="background:#eef6ff; border-left:6px solid #2196f3; padding:16px 20px; border-radius:10px; margin:10px 0; color:#0d2d55;">
  <h2 style="margin:0 0 8px 0; color:#0b3d91;">📊 Dataset</h2>
  <p><b>Source:</b> <a href='https://www.kaggle.com/datasets/bumba5341/advertisingcsv' style='color:#0b3d91;'>Advertising dataset</a> — 200 observations of advertising budgets (in thousands of dollars) for TV, Radio, and Newspaper, with the resulting Sales.</p><table style='border-collapse:collapse; background:#fff; color:#0d2d55;'><tr style='background:#cfe2ff;'><th style='padding:6px 10px; text-align:left;'>Column</th><th style='padding:6px 10px; text-align:left;'>Description</th><th style='padding:6px 10px; text-align:left;'>Type</th></tr><tr><td style='padding:6px 10px;'>TV</td><td style='padding:6px 10px;'>Budget on TV</td><td style='padding:6px 10px;'>feature</td></tr><tr><td style='padding:6px 10px;'>Radio</td><td style='padding:6px 10px;'>Budget on Radio</td><td style='padding:6px 10px;'>feature</td></tr><tr><td style='padding:6px 10px;'>Newspaper</td><td style='padding:6px 10px;'>Budget on Newspaper</td><td style='padding:6px 10px;'>feature</td></tr><tr><td style='padding:6px 10px;'><b>Sales</b></td><td style='padding:6px 10px;'>Product sales</td><td style='padding:6px 10px;'><b>target (continuous)</b></td></tr></table>
</div>
<div style="background:#f3e9ff; border-left:6px solid #7e57c2; padding:16px 20px; border-radius:10px; margin:10px 0; color:#2a0a4a;">
  <h2 style="margin:0 0 8px 0; color:#4a148c;">🎯 Task</h2>
  <p>Predict the continuous target <b>Sales</b> from the three advertising budgets. This is a <b>regression</b> problem; we compare <b>Linear Regression</b>, <b>Ridge</b>, <b>LASSO</b>, evaluated with <b>MSE / RMSE / R²</b>.</p>
</div>


<div style="background-color:#42a5f5; color:#fff; padding:12px 18px; border-radius:10px; margin:10px 0;">
  <h2 style="margin:0; color:#fff;">1️⃣ Import Libraries</h2>
</div>



> First install the extra dependencies (only needed the first time in a fresh environment).

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost lightgbm tqdm -q

clear_output()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

<div style="background-color:#26a69a; color:#fff; padding:12px 18px; border-radius:10px; margin:10px 0;">
  <h2 style="margin:0; color:#fff;">2️⃣ Read the Data</h2>
</div>



In [ ]:
# Download latest version
path = kagglehub.dataset_download("bumba5341/advertisingcsv")

print("Path to dataset files:", path)

In [ ]:
csv_path = os.path.join(path, "Advertising.csv")

df = pd.read_csv(csv_path)
df = df.drop(columns="Unnamed: 0", axis=1)
df.head()

In [ ]:
df.info()

<div style="background-color:#ab47bc; color:#fff; padding:12px 18px; border-radius:10px; margin:10px 0;">
  <h2 style="margin:0; color:#fff;">3️⃣ Exploratory Data Analysis (EDA)</h2>
</div>
<div style="color:#2a2a2a; margin-top:8px;">**Rule of thumb checklist:**

| Question | If YES | If NO |
|----------|--------|-------|
|  **Is the target skewed?** | Consider log transform | Proceed |
|  **Missing values?** | Impute or drop | Proceed |
|  **Categorical columns?** | Encode them | Proceed |
|  **Duplicates?** | Drop them | Proceed |
|  **Different scales?** | Standardize  | Proceed |</div>


In [ ]:
# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Sales")

> **Rule of thumb:** For regression, if the target is heavily skewed, we might apply a log transform. However, for this lab we'll work with the raw values.


In [ ]:
# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# 3. Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

> We don't have categorical columns. If we did, we would need to encode them (convert them into numbers) for our models.

In [ ]:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

> **Rule of thumb:** If duplicates exist, drop them with `df.drop_duplicates(inplace=True)`


In [ ]:
# 5. Do we have different scales in the data?
df.describe()

> **Rule of thumb:** If features have vastly different ranges, then **scale them**. Scaling won't hurt, and it's recommended regardless.
>
> Tree models (RF, LightGBM, CatBoost) don't need scaling, but Linear Regression, Ridge, LASSO, SVM do!
>
> ⚠️ **We do NOT scale here.** Scaling must be fit on the **training set only** and then applied to the test set. We will therefore scale **inside** the cross-validation loop (after the split) to avoid data leakage.


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# We only IMPORT the scaler here. We will fit it on X_train inside the K-Fold loop.
# ❌ DO NOT DO THIS (data leakage):
#     scaler = MinMaxScaler().fit(df[features])
#     df[features] = scaler.transform(df[features])
#
# ✅ Instead: split → fit scaler on X_train → transform X_test (see below).


<div style="background-color:#ef5350; color:#fff; padding:12px 18px; border-radius:10px; margin:10px 0;">
  <h2 style="margin:0; color:#fff;">4️⃣ Training our Regression Models</h2>
</div>
<div style="color:#2a2a2a; margin-top:8px;">> We need to split our data into **X** (features) and **y** (target).</div>


In [ ]:
X = df.drop("Sales", axis=1).astype(float)
y = df['Sales'].astype(float)

---

# Part A: Linear Regression from Scratch

### **Linear Regression Model**

Linear regression predicts a continuous target value by finding the best linear relationship between features and target:

$$\hat{y} = X \cdot \theta$$

Where:
- $X$ = feature matrix (with bias term)
- $\theta$ = weight vector (parameters to learn)
- $\hat{y}$ = predicted values

---

### **Mean Squared Error Loss (MSE)**

We use **Mean Squared Error** to measure how well our predictions match the true values:

$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} (\hat{y}_i - y_i)^2$$

Where:
- $m$ = number of samples
- $y_i$ = true value
- $\hat{y}_i$ = predicted value

> **Intuition:** If we predict the exact value, MSE is 0. The bigger the difference, the higher the penalty (and it's squared, so large errors are penalized heavily)!

In [ ]:
# Mean Squared Error in NumPy
def mean_squared_error(y, y_hat):
  return (1 / (2 * len(y))) * np.sum((y_hat - y) ** 2)

In [ ]:
def normal_equation(X, y):
   # Add bias term (intercept)
    m = X.shape[0]
    X_b = np.c_[np.ones((m, 1)), X]

    # Closed-form solution using pseudoinverse
    theta = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y

    return theta

**Training Linear Regression** using K-Fold Cross-Validation

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score

In [ ]:
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
# Storage for linear regression results for each fold
train_mse_list = []
test_mse_list = []

train_rmse_list = []
test_rmse_list = []

train_r2_list = []
test_r2_list = []

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    # 1) SPLIT first
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 2) SCALE after split: fit on train, transform both train and test
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # 3) Train (closed form)
    theta = normal_equation(X_train_scaled, y_train.values)

    # Add bias column
    X_train_b = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]
    X_test_b  = np.c_[np.ones((X_test_scaled.shape[0], 1)),  X_test_scaled]

    # Predictions
    y_train_pred = X_train_b @ theta
    y_test_pred  = X_test_b @ theta

    # Metrics
    train_mse = sklearn_mse(y_train, y_train_pred)
    test_mse  = sklearn_mse(y_test, y_test_pred)

    train_rmse = np.sqrt(train_mse)
    test_rmse  = np.sqrt(test_mse)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2  = r2_score(y_test, y_test_pred)

    # Store
    train_mse_list.append(train_mse)
    test_mse_list.append(test_mse)

    train_rmse_list.append(train_rmse)
    test_rmse_list.append(test_rmse)

    train_r2_list.append(train_r2)
    test_r2_list.append(test_r2)

    print(f"Train MSE: {train_mse:.4f} | Test MSE: {test_mse:.4f}")
    print(f"Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
    print(f"Train R2: {train_r2:.4f} | Test R2: {test_r2:.4f}")


**Linear Regression Loss Curve**

> use `np.mean(..)` to get the average across all folds.


In [ ]:
print("Linear Regression Results")

print(f"  Average Train MSE: {np.mean(train_mse_list):.4f}")
print(f"  Average Test  MSE: {np.mean(test_mse_list):.4f}")

print(f"  Average Train RMSE: {np.mean(train_rmse_list):.4f}")
print(f"  Average Test  RMSE: {np.mean(test_rmse_list):.4f}")

print(f"  Average Train R2: {np.mean(train_r2_list):.4f}")
print(f"  Average Test  R2: {np.mean(test_r2_list):.4f}")

---

# Part B: Scikit-Learn Regression Models

In [ ]:
from sklearn.linear_model import Ridge, Lasso
#from sklearn.svm import SVR
#from sklearn.tree import DecisionTreeRegressor
#from sklearn.ensemble import RandomForestRegressor
#from lightgbm import LGBMRegressor
#from catboost import CatBoostRegressor

In [ ]:
models = {
  "Ridge Regression": Ridge(alpha=1.0, max_iter=10000),
  "LASSO Regression": Lasso(alpha=1.0,  max_iter=10000),
}

In [ ]:
# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1) SPLIT first
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2) SCALE after split (fit on train only)
  scaler = MinMaxScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_test_scaled  = scaler.transform(X_test)

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)


<div style="background-color:#ff7043; color:#fff; padding:12px 18px; border-radius:10px; margin:10px 0;">
  <h2 style="margin:0; color:#fff;">5️⃣ Results Comparison</h2>
</div>



In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MSE:  {np.mean(all_results[model_name]['mse']):.4f}")
  print(f"  RMSE: {np.mean(all_results[model_name]['rmse']):.4f}")
  print(f"  R2:    {np.mean(all_results[model_name]['r2']):.4f}")

---

## Are our models good?

>Is RMSE = 1.00 good? How to know if a score is good or not?
>
>We need something to compare against. We need a Baseline.
>
>If our models are better than the baseline score, then they are good.

**The simplest baseline we can consider is the mean of the target**.

In [ ]:
# Calculate the baseline predictions (mean of the target)
baseline_pred = np.full_like(y, y.mean())

# Evaluate the baseline
baseline_mse = sklearn_mse(y, baseline_pred)
baseline_rmse = np.sqrt(baseline_mse)
baseline_r2 = r2_score(y, baseline_pred)

print(f"Baseline MSE (using mean target): {baseline_mse:.4f}")
print(f"Baseline RMSE (using mean target): {baseline_rmse:.4f}")
print(f"Baseline R2 (using mean target): {baseline_r2:.4f}")

> **Our models are way better than this baseline because our MSE are lower than the baseline MSE (variance)


<div style="background-color:#66bb6a; color:#fff; padding:12px 18px; border-radius:10px; margin:10px 0;">
  <h2 style="margin:0; color:#fff;">6️⃣ Feature Importance (Ridge & LASSO)</h2>
</div>
<div style="color:#2a2a2a; margin-top:8px;">> For linear models like Ridge and LASSO, the coefficients tell us how important each feature is. Larger absolute values = more important features.

Linear models use coefficients `.coef_` a.k.a weights for each feature</div>


In [ ]:
coeffs = {}

coeffs['Lasso'] = models['LASSO Regression'].coef_
coeffs['Ridge'] = models['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

> **Observation:**
> - LASSO (L1 regularization) pushes some coefficients to exactly 0, effectively performing feature selection (TV).
>
> - Ridge (L2 regularization) smoothly shrinks coefficients but rarely sets them to 0.

<div style="background-color:#ffecd2; padding:14px 20px; border-radius:12px; margin:10px 0; color:#5a2a00;">
  <h2 style="margin:0 0 4px 0; color:#5a2a00;">🎛️ Interactive Playground</h2>
  <p style="margin:0;">Slide the <b>α (alpha)</b> regularization knob and watch the Ridge and LASSO coefficients react in real time!</p>
</div>


In [ ]:
# 🎛️ Interactive: move the slider to see how alpha changes the coefficients
try:
    from ipywidgets import interact, FloatLogSlider
    _HAS_WIDGETS = True
except Exception:
    _HAS_WIDGETS = False

from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
_scaler = MinMaxScaler().fit(X_tr)
X_tr_s, X_te_s = _scaler.transform(X_tr), _scaler.transform(X_te)

def _plot_coefs(alpha=1.0):
    ridge = Ridge(alpha=alpha).fit(X_tr_s, y_tr)
    lasso = Lasso(alpha=alpha, max_iter=10000).fit(X_tr_s, y_tr)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    width = 0.35
    xs = range(len(X.columns))
    ax.bar([x - width/2 for x in xs], ridge.coef_, width=width, label='Ridge', color='#4e8cff')
    ax.bar([x + width/2 for x in xs], lasso.coef_, width=width, label='LASSO', color='#ff6b6b')
    ax.axhline(0, color='gray', lw=0.8)
    ax.set_xticks(list(xs)); ax.set_xticklabels(X.columns)
    ax.set_title(f'Coefficients at alpha = {alpha:.4g}  |  Ridge R²={ridge.score(X_te_s, y_te):.3f}  LASSO R²={lasso.score(X_te_s, y_te):.3f}')
    ax.legend(); plt.tight_layout(); plt.show()

if _HAS_WIDGETS:
    interact(_plot_coefs, alpha=FloatLogSlider(value=1.0, base=10, min=-3, max=3, step=0.1, description='α'))
else:
    # Fallback for environments without ipywidgets
    for a in [0.001, 0.1, 1.0, 10.0, 100.0]:
        _plot_coefs(a)


<div style="background-color:#43cea2; color:#fff; padding:16px 22px; border-radius:12px;  margin:12px 0;">
  <h2 style="margin:0; color:#fff;">🎯 Exercises</h2>
  <p style="margin:6px 0 0 0;">Wherever you see <code style="background:rgba(255,255,255,0.2); padding:1px 5px; border-radius:4px;">None</code>, <code style="background:rgba(255,255,255,0.2); padding:1px 5px; border-radius:4px;">?</code> or <code style="background:rgba(255,255,255,0.2); padding:1px 5px; border-radius:4px;">[?, ?]</code>, fill in the code. Each mini-exercise is one tiny step of the workflow above.</p>
</div>


<div style="background:#fff8e1; border-left:6px solid #fbc02d; padding:14px 18px; border-radius:10px; margin:14px 0; color:#4b3600;">
  <h3 style="margin:0; color:#5d4100;">🧩 Exercise 1 — Split the data</h3>
  <p style="margin:8px 0 0 0;">Complete the call to <code>train_test_split</code> so that <b>20% of the samples</b> go to the test set, and the split is <b>reproducible</b> (use <code>random_state=42</code>).</p>
</div>


In [ ]:
from sklearn.model_selection import train_test_split

# TODO: fill in the ? so that 20% goes to test and the split is reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=?,        # ← ?
    random_state=?      # ← ?
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)


<div style="background:#fff8e1; border-left:6px solid #fbc02d; padding:14px 18px; border-radius:10px; margin:14px 0; color:#4b3600;">
  <h3 style="margin:0; color:#5d4100;">🧩 Exercise 2 — Scale <i>after</i> the split (no leakage!)</h3>
  <p style="margin:8px 0 0 0;">Complete the next cell so the scaler is <b>fit on the training set only</b>, then <b>applied</b> to the test set.</p>
</div>


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# TODO: learn the mean/std from X_train AND scale it in one call.
X_train_scaled = None   # ← hint: scaler.fit_transform(...)

# TODO: use the SAME scaler (already fit on train) to transform X_test.
X_test_scaled  = None   # ← hint: scaler.transform(...)

print("mean(X_train_scaled) ≈ 0 ?", X_train_scaled.mean(axis=0).round(3))
print("std (X_train_scaled) ≈ 1 ?", X_train_scaled.std(axis=0).round(3))


<div style="background:#fff8e1; border-left:6px solid #fbc02d; padding:14px 18px; border-radius:10px; margin:14px 0; color:#4b3600;">
  <h3 style="margin:0; color:#5d4100;">🧩 Exercise 3 — Train a Ridge model and compute RMSE</h3>
  <p style="margin:8px 0 0 0;">Fill the blanks: instantiate <code>Ridge(alpha=1.0)</code>, fit it on the scaled training data, and compute the <b>RMSE</b> on the test set.</p>
</div>


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error as sklearn_mse

# 1) Instantiate the Ridge model with alpha = 1.0
ridge = None   # ← TODO

# 2) Fit on the training set
None           # ← TODO: ridge.fit( ? , ? )

# 3) Predict on the test set
y_pred = None  # ← TODO

# 4) Compute RMSE = sqrt( MSE(y_test, y_pred) )
rmse = None    # ← TODO

print(f"Ridge test RMSE: {rmse:.4f}")


<div style="background:#fff8e1; border-left:6px solid #fbc02d; padding:14px 18px; border-radius:10px; margin:14px 0; color:#4b3600;">
  <h3 style="margin:0; color:#5d4100;">🧩 Exercise 4 — Use <code>RidgeCV</code> / <code>LassoCV</code></h3>
  <p style="margin:8px 0 0 0;">Instead of guessing <code>alpha</code>, <code>scikit-learn</code> provides cross-validated versions. Complete the code below.</p>
</div>


In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV

alphas = [0.01, 0.1, 1.0, 10.0, 100.0]

# TODO: create a RidgeCV that tries each of the alphas above with 5-fold CV
ridge_cv = RidgeCV(alphas=?, cv=?)

# TODO: create a LassoCV with the same alphas and 5-fold CV, max_iter=10000
lasso_cv = LassoCV(alphas=?, cv=?, max_iter=10000, random_state=42)

ridge_cv.fit(X_train_scaled, y_train)
lasso_cv.fit(X_train_scaled, y_train)

print(f"Best alpha (Ridge): {ridge_cv.alpha_}")
print(f"Best alpha (Lasso): {lasso_cv.alpha_}")


### **Contribution: Sattam Altwaim** :)